# Set up

In [ ]:
import pandas as pd
import os
import matplotlib as plt

from surprise import Reader, AlgoBase, Dataset, accuracy
from surprise.model_selection import train_test_split
import numpy as np
from collections import defaultdict
import recmetrics
from sklearn.metrics import ndcg_score

In [2]:
SEED = 42; TOP_K = 10; RELEVANCE_THRESHOLD = 7; MIN_ITEM_RATINGS = 50
N_USERS = 5000; N_USERS_COVERAGE = 300

np.random.seed(SEED)
ratings_df = pd.read_csv("../processed_data/explicit_ratings.csv")
all_users = ratings_df["user_id"].unique()
np.random.shuffle(all_users)

reader      = Reader(rating_scale=(1, 10))
reader_wide = Reader(rating_scale=(1, 20))

sample = ratings_df[ratings_df["user_id"].isin(all_users[:N_USERS])][["user_id", "anime_id", "rating"]]
data = Dataset.load_from_df(sample, reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=SEED)

sample_cov = ratings_df[ratings_df["user_id"].isin(all_users[:N_USERS_COVERAGE])][["user_id", "anime_id", "rating"]]
data_cov = Dataset.load_from_df(sample_cov, reader_wide)
trainset_cov = data_cov.build_full_trainset()
antitestset_cov = trainset_cov.build_anti_testset()
catalog = [trainset_cov.to_raw_iid(iid) for iid in trainset_cov.all_items()]

genre_cols = ['Action', 'Adventure', 'Cars', 'Comedy', 'Dementia', 'Demons', 'Drama', 'Ecchi', 'Fantasy', 'Game', 'Harem', 'Historical', 'Horror', 'Josei', 'Kids', 'Magic', 'Martial Arts', 'Mecha', 'Military', 'Music', 'Mystery', 'Parody', 'Police', 'Psychological', 'Romance', 'Samurai', 'School', 'Sci-Fi', 'Seinen', 'Shoujo', 'Shoujo Ai', 'Shounen', 'Shounen Ai', 'Slice of Life', 'Space', 'Sports', 'Super Power', 'Supernatural', 'Thriller', 'Vampire', 'Yaoi']
anime_meta = pd.read_csv("../processed_data/anime_processed.csv")[["MAL_ID"] + genre_cols].set_index("MAL_ID")

In [3]:
import gc

def get_top_n(predictions, n=TOP_K):
    user_preds = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_preds[uid].append((iid, est, true_r))
    return {uid: sorted(v, key=lambda x: x[1], reverse=True)[:n] for uid, v in user_preds.items()}

def eval_accuracy(predictions, n=TOP_K, threshold=RELEVANCE_THRESHOLD):
    rmse = accuracy.rmse(predictions, verbose=False)
    mae  = accuracy.mae(predictions, verbose=False)
    user_preds = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_preds[uid].append((iid, est, true_r))
    precisions, recalls, ndcgs = [], [], []
    for uid, preds in user_preds.items():
        top = sorted(preds, key=lambda x: x[1], reverse=True)[:n]
        hits      = sum(r >= threshold for _, _, r in top)
        total_rel = sum(r >= threshold for _, _, r in preds)
        precisions.append(hits / n)
        recalls.append(hits / total_rel if total_rel > 0 else 0)
        true_r = [r for _, _, r in top]; est_r = [e for _, e, _ in top]
        if len(true_r) > 1:
            ndcgs.append(ndcg_score([true_r], [est_r]))
    return {"RMSE": round(rmse, 4), "MAE": round(mae, 4),
            f"Precision@{n}": round(np.mean(precisions), 4),
            f"Recall@{n}":    round(np.mean(recalls), 4),
            f"NDCG@{n}":      round(np.mean(ndcgs), 4)}

def eval_coverage(predictions, catalog, feature_df, n=TOP_K):
    top_n     = get_top_n(predictions, n)
    full_recs = [[iid for iid, _, _ in v] for v in top_n.values() if len(v) == n]
    cov  = recmetrics.prediction_coverage(full_recs, catalog) if full_recs else 0.0
    pers = recmetrics.personalization(full_recs)              if full_recs else 0.0
    ils  = recmetrics.intra_list_similarity(full_recs, feature_df)
    return {"Coverage": round(cov, 4), "Personalization": round(pers, 4),
            "Intra-List Similarity": round(ils, 4)}

# Non-Personalized Recommenders

## Popular Recommender

In [4]:
class PopularRecommender(AlgoBase):
    def __init__(self, min_ratings=MIN_ITEM_RATINGS):
        AlgoBase.__init__(self)
        self.min_ratings = min_ratings

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        df = pd.DataFrame([(i, r) for (_, i, r) in trainset.all_ratings()], columns=["item", "rating"])
        stats = df.groupby("item").agg(count=("rating", "count"), mean=("rating", "mean"))
        self.item_means = stats.loc[stats["count"] >= self.min_ratings, "mean"]
        self.global_mean = trainset.global_mean
        return self

    def estimate(self, u, i):
        if i in self.item_means.index:
            return self.item_means[i]
        return self.global_mean

In [5]:
popular = PopularRecommender()

popular.fit(trainset)
acc = eval_accuracy(popular.test(testset))

popular.fit(trainset_cov)
cov_preds = popular.test(antitestset_cov)
cov = eval_coverage(cov_preds, catalog, anime_meta)
del cov_preds; gc.collect()

pd.Series({**acc, **cov})

RMSE                     1.6112
MAE                      1.2166
Precision@10             0.8144
Recall@10                0.5219
NDCG@10                  0.9735
Coverage                 0.9900
Personalization          0.4186
Intra-List Similarity    0.2964
dtype: float64

## Random Recommender

In [6]:
class RandomRecommender(AlgoBase):
    def __init__(self):
        AlgoBase.__init__(self)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        ratings = [r for (_, _, r) in trainset.all_ratings()]
        self.mean = np.mean(ratings)
        self.std  = np.std(ratings)
        return self

    def estimate(self, u, i):
        return np.random.normal(self.mean, self.std)

In [7]:
random_rec = RandomRecommender()

random_rec.fit(trainset)
acc = eval_accuracy(random_rec.test(testset))

random_rec.fit(trainset_cov)
cov_preds = random_rec.test(antitestset_cov)
cov = eval_coverage(cov_preds, catalog, anime_meta)
del cov_preds; gc.collect()

pd.Series({**acc, **cov})

RMSE                      2.4038
MAE                       1.9041
Precision@10              0.7183
Recall@10                 0.4844
NDCG@10                   0.9472
Coverage                 38.9800
Personalization           0.9984
Intra-List Similarity     0.1925
dtype: float64

# Content-Based Recommenders